In [1]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_53087/2821732259.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import AsyncHtmlLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# Splitter to generate parent coarse chunks from original documents (parsed from web pages)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)

# Splitter to generate child granular chunks from parent coarse chunks
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

EMBEDDING_MODEL = 'nomic-embed-text:latest'
# Vector store collection to host child granular chunks
child_chunks_collection = Chroma(
    collection_name="uk_child_chunks",
    embedding_function=OllamaEmbeddings(model=EMBEDDING_MODEL),
)
child_chunks_collection.reset_collection()
doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

# Retriever to link parent coarse chunks to child granular chunks
multi_vector_retriever = MultiVectorRetriever(
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)

In [3]:
from langchain_community.document_transformers import Html2TextTransformer

uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]
html2text_transformer = Html2TextTransformer()

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]) #F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id #G

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(all_granular_chunks)
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks)))
    

Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  5.93it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  5.68it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  6.34it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  9.80it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|##################################################| 1/1 [00:00<00:00,  6.79it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [4]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
print(retrieved_docs[0])

page_content='**Go Cornwall Bus** buses operate between:

  * **10** \- Plymouth to Saltash, Looe and Polperro
  * **11** \- Plymouth to Saltash, Liskeard, Bodmin, Wadebridge and Padstow

**Stagecoach** buses operate between Barnstaple, Holsworthy, Launceston and
Tavistock, across the Cornwall and Devon border (**85**).

## Get around

[edit]

### By bus

[edit]

Thanks to Transport for Cornwall, all bus tickets are interchangeable across
the different companies (except certain town buses in St Ives and Fowey). The
**Cornwall All Day ticket** allows unlimited travel for a calendar day. As of
2025, day passes are £8 for adults and £5 for under-19s and £3 singles,
regardless of age. Payment is by cash or contactless. Real time information
and timetables can now be found through Transport for Cornwall (most reliable
for real time info), Transit (flaky & unreliable at times, and only buses) or
Citymapper (shows all public transport in Cornwall, yet sometimes unreliable
for real time info).

In [5]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")
print(child_docs_only[0])

page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and
Plymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for
under-16s.

There is also a new Pay as you Go smartcard system for Cornwall, run by GWR
but valid on all trains in Cornwall.

### By ferry/boat

[edit]' metadata={'doc_id': 'a1e0ad1b-4ff7-4965-9d60-879ce76891c6', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/Cornwall'}
